<h1>Coding Session #3 - Convolutional Neural Network</h1>

Führen Sie folgende Zelle aus, um alle benötigten Bibliotheken zu installieren:

In [ ]:
!pip install -r requirements.txt

## Ziel

In dieser Coding Session soll ein Faltungsnetz (Convolutional Neural Network - CNN) implementiert und trainiert werden. CCNs sind besonders effizient für die Bild- oder Videoverarbeitung. Da es sich bei Bildern um hochdimensionale Inputs handelt, sind Fully Connected (FC) Layer ungeeignet.

## 1 Convolutional Layer

Um ein RGB-Bild mit $1,920\times 1,080$ Pixeln zu repräsentieren, bräuchte man einen FC Layer mit 6.2 Mio. Neuronen:

<img src="images/curse_of_dimensionality.png" width="600px"></img>

Hier schafft das CNN Abhilfe. Dieses nutzt Convolutional Layer. Jeder Convolutional Layer hat eine bestimmte Anzahl an Kerneln, mit denen Faltungsoperationen auf den Input angewendet werden. Dafür slidet der Kernel Spalte für Spalte und Zeile für Zeile über das Input-Bild. An jeder Stelle werden die Pixel der Region des Inputbildes $X$ elementweise mit den Gewichtungen des Kernels $K$ multipliziert und anschließend die Summe der Produkte gebildet, um den Output an der Stelle $Y[m, n]$ zu erhalten. 

<img src="images/convolution001.png" width="600px"></img>

Mit Hilfe von Faltungen lassen sich Merkmale (Features) wie Ecken oder Kanten im Bild erkennen. Die Gewichtungen der Kernels sind lernbar. Somit kann das neuronale Netz daraufhin optimiert werden, aussagekräftige Features für den Input zu extrahieren.

Folgendes Beispiel visualisiert eine Faltung mit einem Kernel zur Detektion von vertikalen Kanten:

<img src="images/conv_animation.gif" width="600px"></img>

> <span style="color:#00A1E3">**Aufgabe 1 - Convolutional Layer**</span>
> 1. Erstellen Sie eine Klasse `ConvLayer`, welche von `Module` ableitet.
> 2. Erstellen Sie den Konstruktor (`def __init__(self, in_channels, out_channels, kernel_size)`).
>       - Interpretieren Sie `in_channels` als $C_{in}$, `out_channels` als $C_{out}$ und `kernel_size` als $K$.
>       - Die Kernelgröße ist damit $(K \times K)$.
>       - Speichern Sie die Parameter in `self.in_channels`, `self.out_channels` und `self.kernel_size`.
>       - Nutzen Sie He-Initialisierung mit $fan_{in}=C_{in}\cdot K\cdot K$.
>       - Initialisieren Sie die Gewichte $W\in\mathbb{R}^{C_{out}\times C_{in}\times K\times K}$ (`self.kernels`) über `np.random.randn(...) * np.sqrt(2.0 / fan_in)`.
>       - Initialisieren Sie Biases $b\in\mathbb{R}^{C_{out}}$ (ein Bias pro Output-Kanal), z.B. mit Nullen: `self.biases = np.zeros((out_channels, 1))`.
> 3. Implementieren Sie die Vorwärtspropagation `forward(self, input:np.ndarray)`.
>       - Der Input $X$ (`input`) liegt in der Form $(B, C_{in}, H_{in}, W_{in})$. Speichern Sie diesen in `self.input`.
>       - Berechnen Sie $H_{out}=H_{in}-K+1$ und $W_{out}=W_{in}-K+1$.
>       - Initialisieren Sie den Output $Y$ (entspricht `output`) mit Form $(B, C_{out}, H_{out}, W_{out})$.
>       - Berechnen Sie die Faltung für alle Batch-, Kanal- und Ortsindizes:
>         $$Y[b,o,i,j] = \sum_{c=1}^{C_{in}} \sum_{u=0}^{K-1} \sum_{v=0}^{K-1} X[b,c,i+u,j+v]\cdot W[o,c,u,v] + b[o]$$
>       - Geben Sie $Y$ (`output`) zurück.
> 4. Implementieren Sie die Rückwärtspropagation `backward(self, grad_output:np.ndarray)`.
>       - Bezeichnen Sie `grad_output` als $\nabla Y$ mit Form $(B, C_{out}, H_{out}, W_{out})$.
>       - Initialisieren Sie $\nabla W$ (`grad_kernels`) und $\nabla X$ (`grad_input`) mit Nullen in den passenden Formen (`np.zeros_like(self.kernels)`, `np.zeros_like(self.input)`).
>       - **Warum Akkumulation?** Ein Gewicht bzw. ein Input-Pixel beeinflusst viele Outputs. Deshalb werden Beiträge aus allen relevanten Positionen aufsummiert.
>       - **Gradient für die Kernelgewichte** (entspricht `grad_kernels[oc, ic] += ...`):
>         $$\nabla W[o,c,u,v] = \sum_{b=1}^{B}\sum_{i=0}^{H_{out}-1}\sum_{j=0}^{W_{out}-1} X[b,c,i+u,j+v]\cdot \nabla Y[b,o,i,j]$$
>         Im Code für feste `(oc, ic, i, j)`:
>         - `region = self.input[:, ic, i:i+K, j:j+K]` entspricht $X[:,c,i:i+K,j:j+K]$ und hat Form $(B,K,K)$.
>         - `grad_output[:, oc, i, j]` hat Form $(B,)$; mit `[:, np.newaxis, np.newaxis]` wird es zu $(B,1,1)$ erweitert und per Broadcast auf $(B,K,K)$ gebracht.
>         - `region * grad_output[:, oc, i, j][:, np.newaxis, np.newaxis]` liefert die Beiträge pro Sample und pro Kernel-Position $(u,v)$ in Form $(B,K,K)$.
>         - `np.sum(..., axis=0)` summiert uber die Batch-Dimension $b$ und ergibt ein $(K,K)$-Array fur genau dieses `(oc, ic)` und diese Position $(i, j)$.
>         - `grad_kernels[oc, ic] += ...` akkumuliert diese Beitraege uber alle `(i, j)`-Positionen.
>       - **Gradient für den Input** (entspricht `grad_input[:, ic, i:i+K, j:j+K] += ...`):
>         $$\nabla X[b,c,p,q] = \sum_{o=1}^{C_{out}}\sum_{i,j:\,p=i+u,\,q=j+v} W[o,c,u,v]\cdot \nabla Y[b,o,i,j]$$
>         Äquivalent als lokale Update-Regel pro `(oc, ic, i, j)` (Matrixmultiplikation $\to$ effizienter):
>         $$\nabla X[:,c,i:i+K,j:j+K] \mathrel{+}= W[o,c,:,:]\cdot \nabla Y[:,o,i,j]$$
>           - Dabei sorgt `+=` für die notwendige Überlagerung (Akkumulation) aus allen überlappenden Rezeptivfeldern.
>           - `:` auf erster Dimension $\to$ Operation über gesamten Batch (effizienter als Schleife)
>       - Führen Sie ein Update der Gewichte durch: $W \leftarrow W - \nabla W$ (im Code: `self.kernels -= grad_kernels`).
>       - Geben Sie $\nabla X$ (`grad_input`) zurück.

In [ ]:
import numpy as np
from model import Module

# Your code here


> <span style="color:#00A1E3">**Aufgabe 2 - MaxPool Layer**</span>
> 1. Erstellen Sie eine Klasse `MaxPool`, welche von `Module` ableitet.
> 2. Erstellen Sie den Konstruktor (`def __init__(self, kernel_size, stride)`).
>       - Interpretieren Sie `kernel_size` als Poolgröße $K$ und `stride` als Schrittweite $S$.
>       - Speichern Sie beide Parameter in `self.kernel_size` und `self.stride`.
> 3. Implementieren Sie die Vorwärtspropagation `forward(self, input:np.ndarray)`.
>       - Der Input $X$ (`input`) liegt in der Form $(B, C, H_{in}, W_{in})$. Speichern Sie ihn in `self.input`.
>       - Berechnen Sie die Output-Dimensionen mit:
>         $$H_{out}=\left\lfloor\frac{H_{in}-K}{S}\right\rfloor+1,\qquad W_{out}=\left\lfloor\frac{W_{in}-K}{S}\right\rfloor+1$$
>       - Initialisieren Sie den Output $Y$ (`output`) mit Form $(B, C, H_{out}, W_{out})$.
>       - Für jedes Fenster gilt:
>         $$Y[b,c,i,j]=\max_{0\le u,v<K} X[b,c,i\cdot S+u,\,j\cdot S+v]$$
>       - Geben Sie $Y$ (`output`) zurück.
> 4. Implementieren Sie die Rückwärtspropagation `backward(self, grad_output:np.ndarray)`.
>       - Bezeichnen Sie `grad_output` als $\nabla Y$ mit Form $(B, C, H_{out}, W_{out})$.
>       - Initialisieren Sie $\nabla X$ (`grad_input`) mit Nullen und Form wie `self.input`.
>       - Für jedes Pooling-Fenster wird der Gradient nur an die Position des Maximums weitergegeben:
>         $$\nabla X[b,c,i\cdot S+u^*,j\cdot S+v^*] \mathrel{+}= \nabla Y[b,c,i,j]$$
>         wobei $(u^*,v^*)=\arg\max_{0\le u,v<K} X[b,c,i\cdot S+u,\,j\cdot S+v]$.
>       - Nutzen Sie im Code `argmax` pro Fenster (wie in `max_indices`) und mappen Sie den flachen Index zurück auf `(max_i, max_j)`.
>       - Geben Sie $\nabla X$ (`grad_input`) zurück.

In [ ]:
import numpy as np
from model import Module

# Your code here


Ein CNN soll lernen, folgende Bilde zu klassifizieren:

In [ ]:
from data import CornersAndEdgesDataset
import visualization as vis


ds = CornersAndEdgesDataset(num_samples=1000, image_size=256)


labels_txt = [f"{i} - {ds.LABELS[label]}" for i, label in enumerate(ds.labels)]

vis.plot_images(ds.images, labels_txt)

Führen Sie folgende Zelle aus, um ein CNN zu trainieren und die Ergebnisse zu visualisieren:

In [ ]:
from data import CornersAndEdgesDataset, one_hot_encode
from metrics import compute_accuracy, CrossEntropyLossWithSoftmax
from model import NeuralNetwork, ReLU, Flatten, DenseLayer
from tqdm import tqdm
import visualization as vis
import numpy as np

EPOCHS          = 20
BATCH_SIZE      = 32
LEARNING_RATE   = 0.1
IMAGE_SIZE      = 8
VIS_KERNELS     = 8

# Demo-Modus für klar sichtbare Edge/Corner-Kernels im 1. Conv-Layer
ds = CornersAndEdgesDataset(
    num_samples=50_000,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    augment=False,
    seed=42
 )

model = NeuralNetwork(modules=[
    MaxPool(kernel_size=4, stride=4),
    ConvLayer(in_channels=1, out_channels=8, kernel_size=2, learn_biases=False),
    Flatten()
])


model.print_shapes(ds.images[0:1])

criterion = CrossEntropyLossWithSoftmax()
dense_layers = [m for m in model.modules if isinstance(m, DenseLayer)]
conv_layers = [m for m in model.modules if hasattr(m, "kernels")]

# Feste Referenz für den visuellen Vergleich
sample_image = ds.images[0, 0].copy()

print("Vor Training: First Layer Kernels")
vis.visualize_first_layer_kernels(model, max_kernels=VIS_KERNELS)

# Training
for epoch in range(EPOCHS):
    losses_train, accs_train = [], []
    prog_train = tqdm(ds, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for X, Y in prog_train:
        Y = one_hot_encode(Y, num_classes=len(ds.LABELS))

        Y_hat = model.forward(X)
        loss = criterion.forward(Y_hat, Y)

        acc = compute_accuracy(Y_hat, Y)


        losses_train.append(loss)
        accs_train.append(acc)


        grad_outputs = criterion.backward()
        model.backward(grad_outputs * LEARNING_RATE)

        prog_train.set_description(
            f"Epoch {epoch+1}/{EPOCHS}, Loss: {np.mean(losses_train):.4f}, Accuracy: {np.mean(accs_train):.4f}"
        )


    print(
        f"Epoch {epoch+1}/{EPOCHS}, Loss: {np.mean(losses_train):.4f}, "
        f"Accuracy: {np.mean(accs_train):.4f}"
    )

print("Nach Training: First Layer Kernels")
vis.visualize_first_layer_kernels(model, max_kernels=VIS_KERNELS)

print("Nach Training: Feature Maps (gleiches Preprocessing wie im Training)")
for image in ds.images:
    vis.visualize_features(image[0], model, n_kernels=VIS_KERNELS, zscore_input=False)